# Famous Percentage Analysis on Facebook/Meta Users

CREATE TABLE famous (user_id INT, follower_id INT);

INSERT INTO famous VALUES
(1, 2), (1, 3), (2, 4), (5, 1), (5, 3), 
(11, 7), (12, 8), (13, 5), (13, 10), 
(14, 12), (14, 3), (15, 14), (15, 13);


A table named “famous” has two columns called user id and follower id. It represents each user ID has a particular follower ID. These follower IDs are also users of hashtag#Facebook / hashtag#Meta. Then, find the famous percentage of each user. 

Famous Percentage = number of followers a user has / total number of users on the platform.


In [0]:
%skip
CREATE TABLE ska_catalog.bronze.famous (user_id INT, follower_id INT);
INSERT INTO ska_catalog.bronze.famous VALUES
(1, 2), (1, 3), (2, 4), (5, 1), (5, 3), 
(11, 7), (12, 8), (13, 5), (13, 10), 
(14, 12), (14, 3), (15, 14), (15, 13);

In [0]:
-- Selecting all users in the table.
SELECT user_id AS id FROM ska_catalog.bronze.famous
UNION
SELECT follower_id AS id FROM ska_catalog.bronze.famous

In [0]:
-- followers per user_id
SELECT user_id, COUNT(follower_id) FROM ska_catalog.bronze.famous
GROUP BY user_id

In [0]:
With all_users AS (
  SELECT user_id AS id FROM ska_catalog.bronze.famous
  UNION
  SELECT follower_id AS id FROM ska_catalog.bronze.famous
),
followers_per_user AS (
  SELECT user_id , COUNT(follower_id) AS `followers` 
  FROM ska_catalog.bronze.famous
  GROUP BY user_id
)
SELECT f.user_id,
ROUND(f.followers * 100.0 / (SELECT COUNT(*) FROM all_users), 4) AS followers_pct
FROM followers_per_user f
ORDER BY  f.user_id;

In [0]:
SELECT * FROM ska_catalog.bronze.famous

In [0]:
%skip
INSERT INTO ska_catalog.bronze.famous
VALUES (15, 16),(1,17), (15,17),(3,20)

In [0]:
-- Top Three follower per user
SELECT user_id, COUNT(follower_id) AS `FOLLOWERS_COUNT` 
FROM ska_catalog.bronze.famous
GROUP BY user_id
ORDER BY FOLLOWERS_COUNT DESC
LIMIT 3;

In [0]:
-- Which users don’t have any followers?
-- This users will be a follower but not the user.
SELECT DISTINCT follower_id AS user_without_followers
FROM ska_catalog.bronze.famous
WHERE follower_id NOT IN (SELECT DISTINCT user_id FROM ska_catalog.bronze.famous)

In [0]:
-- Which users follow the most other users?
SELECT follower_id, COUNT(USER_ID) as followers_count FROM ska_catalog.bronze.famous
GROUP BY follower_id
ORDER BY followers_count DESC

In [0]:
-- Find mutual followers (users who follow each other).
SELECT a.user_id AS user1, a.follower_id AS user2
FROM ska_catalog.bronze.famous a
JOIN ska_catalog.bronze.famous b
  ON a.user_id = b.follower_id AND a.follower_id = b.user_id AND a.user_id < b.user_id

Find users with the highest follower-to-following ratio.

Requires calculating both followers and following counts per user.

- 1. Find the no of followers a user_id has. (followers)
- 2. Find the no of users a follower_id follows (following)
- 3. Combine both using collease.
- 4. Find the ratio using followers/following

In [0]:
WITH followers AS (
  SELECT user_id, COUNT(follower_id) AS followers_count
  FROM ska_catalog.bronze.famous
  GROUP BY user_id
),
following AS (
  SELECT follower_id as `user_id`, COUNT(user_id) AS following_count
  FROM ska_catalog.bronze.famous
  GROUP BY  follower_id
),
combined AS (
  SELECT COALESCE(f.user_id, fw.user_id,0) AS user_id,
        COALESCE(f.followers_count,0) AS followers,
        COALESCE(fw.following_count,0) AS following 
  FROM followers f
  FULL OUTER JOIN following fw
  ON f.user_id = fw.user_id
)
SELECT  user_id,
        followers,
        following,
        CASE
            WHEN following = 0 THEN followers
            ELSE ROUND(followers * 1.0 / following, 2)
        END AS ratio
FROM combined
ORDER BY user_id ASC,ratio DESC;

In [0]:
-- Find users who follow only one person.
SELECT follower_id , COUNT(*) FROM ska_catalog.bronze.famous
GROUP BY follower_id
HAVING COUNT(*) = 1

Find users who are followed by users with many followers.

This explores **second-degree influence**.

In [0]:
WITH followers_cnt AS (
  SELECT user_id, COUNT(follower_id) as `follower_cnt`
  FROM ska_catalog.bronze.famous
  GROUP BY user_id
),
influential_user AS (
  SELECT user_id FROM followers_cnt
  WHERE follower_cnt >= 2
)
SELECT DISTINCT follower_id AS FOLLOWERS FROM ska_catalog.bronze.famous
WHERE follower_id IN (SELECT user_id FROM influential_user)